## Plots for the Paper

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import polars as pl
from r_utils import fit_lmer_with_vc, fit_lmer, fit_lmer_full, format_term
from plot_utils import _configure_fonts, plot_emmeans_grouped_bar, SETTING_COLOR, SETTING_MAP, GRAPH_MAP, SETTING_MARKER, OKABE_ITO
import matplotlib.pyplot as plt

_configure_fonts()
# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
result_path = Path.cwd().parent / "data" / "analysis" 
COL_PRETTY_NAMES = {
    "plasticity_tv": "Plasticity",
    "monotonicity": "Directedness",
    "influence_out": "Outgoing Influence",
    'modal_consensus_change': 'Consensus Change',
    "modal_consensus_T": "Consensus",
}

## Contrasts

In [ ]:
agent_contrast = pl.read_csv(result_path / "contrasts" / "agent" / "contrast_summary.csv")
run_contrast = pl.read_csv(result_path / "contrasts" / "runs" / "contrast_summary.csv")
contrasts = pd.concat([agent_contrast.to_pandas(), run_contrast.to_pandas()], ignore_index=True)
contrasts.to_csv(result_path / "contrasts" / "combined_contrasts.csv", index=False)
contrasts.sample(10)

In [ ]:
from tkinter import font

from matplotlib.lines import Line2D

from notebooks.plot_utils import GRAPH_COLOR

def plot_contrast_forest(
    summary,
    contrasts_order=None,
    outcomes_order=None,
    graph_types_order=("erdos-renyi", "watts-strogatz"),
    use_standardized=True,
    figsize=(14, 4.8),
):
    """Forest plot with one panel per contrast and 3 estimates per outcome.

    For each outcome row inside each contrast panel, this plots a triplet of
    estimates with CIs:
      - Marginal (pooled)
      - Within Erdos-Renyi
      - Within Watts-Strogatz

    Args:
        summary: DataFrame from build_contrast_summary.
        contrasts_order: Ordered list of contrasts to display as panels.
        outcomes_order: Ordered list of outcomes to display on y-axis.
        graph_types_order: Order of graph labels used for within contrasts.
        use_standardized: Whether to use standardized estimates/CIs.
        figsize: Matplotlib figure size.

    Returns:
        Matplotlib figure.
    """
    if contrasts_order is None:
        contrasts_order = ["role_eff", "model_eff", "alignment", "dissociation"]
    if outcomes_order is None:
        outcomes_order = summary["outcome"].unique().tolist()

    contrast_labels = {
        "role_eff": "Role Effect",
        "model_eff": "Model Effect",
        "alignment": "Role Alignment Effect",
        "dissociation": "Heterogeneity Effect",
    }

    triplet_specs = [
        ("marginal", None, "Marginal (pooled)", OKABE_ITO['blue'], 0.0),
        ("within", graph_types_order[0], "Erdos-Renyi", GRAPH_COLOR['erdos-renyi'], -0.23),
        ("within", graph_types_order[1], "Watts-Strogatz", GRAPH_COLOR['watts-strogatz'], 0.23),
    ]

    est_col = "std_estimate" if use_standardized else "estimate"
    lo_col = "std_lower" if use_standardized else "lower"
    hi_col = "std_upper" if use_standardized else "upper"

    fig, axes = plt.subplots(
        1,
        len(contrasts_order),
        figsize=figsize,
        sharey=True,
        sharex=True,
        squeeze=False,
    )
    axes = axes[0]

    y_positions = {o: i for i, o in enumerate(outcomes_order)}

    for panel_idx, (ax, contrast_name) in enumerate(zip(axes, contrasts_order)):
        sub_contrast = summary[summary["contrast"] == contrast_name]

        for y in range(len(outcomes_order)):
            ax.axhline(y+0.5, color="0.88", linewidth=0.7, alpha=0.7, zorder=0)

        for outcome in outcomes_order:
            if outcome not in y_positions:
                continue
            y_base = y_positions[outcome]

            for ctype, gtype, _, color, y_offset in triplet_specs:
                mask = sub_contrast["contrast_type"] == ctype
                if gtype is None:
                    mask &= sub_contrast["graph_type"].isna()
                else:
                    mask &= sub_contrast["graph_type"] == gtype
                mask &= sub_contrast["outcome"] == outcome

                sub = sub_contrast[mask]
                if sub.empty:
                    continue

                row = sub.iloc[0]
                is_sig = bool(row["sig"])
                alpha = 1.0 if is_sig else 0.45
                marker_face = color if is_sig else "white"

                ax.errorbar(
                    row[est_col],
                    y_base + y_offset,
                    xerr=[[row[est_col] - row[lo_col]], [row[hi_col] - row[est_col]]],
                    fmt="o",
                    color=color,
                    markerfacecolor=marker_face,
                    markeredgecolor=color,
                    capsize=3,
                    markersize=3,
                    alpha=alpha,
                    linewidth=1.5,
                )

        ax.axvline(0, color="k", linestyle="--", alpha=0.35, linewidth=1)
        ax.spines[["top", "right"]].set_visible(False)
        panel_label = chr(ord("A") + panel_idx)
        ax.set_title(
            f"({panel_label}) {contrast_labels.get(contrast_name, contrast_name)}",
            fontsize=13,
        )
        ax.set_xlabel("Cohen's d" if use_standardized else "Effect estimate", fontsize=12)
        
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_linewidth(1.5)
        ax.spines["bottom"].set_linewidth(1.5)
        ax.tick_params(width=1.3, length=5)

    axes[0].set_yticks(range(len(outcomes_order)))
    axes[0].set_yticklabels([COL_PRETTY_NAMES.get(o, o) for o in outcomes_order], fontsize=12)
    

    legend_handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            color=color,
            markerfacecolor=color,
            markeredgecolor=color,
            linestyle="None",
            markersize=6,
            label=label,
        )
        for _, _, label, color, _ in triplet_specs
    ]
    fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=3,
        frameon=False,
        fontsize=12,
        bbox_to_anchor=(0.5, -0.07),
    )

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    return fig

In [ ]:
contrasts_latex = pl.from_pandas(contrasts[contrasts['outcome'].isin(COL_PRETTY_NAMES.keys()) & contrasts['contrast_type'].isin(['marginal', 'within'])])

#### Marginal

In [ ]:
# contrast_labels = {
#         "role_eff": "Role Effect",
#         "model_eff": "Model Effect",
#         "alignment": "Role Alignment",
#         "dissociation": "Heterogeneity",
#     }

# def fmt_p(p: float) -> str:
#     if pd.isna(p):
#         return ""
#     return "<0.001" if p < 0.001 else f"{p:.3f}"


# lfs  = (contrasts_latex.filter(pl.col('contrast_type') == 'marginal')
#  .with_columns(pl.col('outcome').map_elements(lambda x: COL_PRETTY_NAMES.get(x, x)).alias('Outcome'))
#  .with_columns(pl.col('contrast').map_elements(lambda x: contrast_labels.get(x, x)).alias('Contrast'))
#  .select(
#      pl.col('Outcome'),
#      pl.col('Contrast'),
#      pl.col('estimate').round(3),
#      pl.col('lower').round(3),
#      pl.col('upper').round(3),
#      pl.col('SE').round(3),
#      pl.col('t').round(3),
#      pl.col('p').map_elements(fmt_p).alias('p'),
#      pl.col('df').round(3),
#  )
#  ).sort(['Outcome', 'Contrast']).to_pandas()
# lfs['Contrast'] = pd.Categorical(lfs['Contrast'], categories=['Role Effect', 'Model Effect', 'Role Alignment', 'Heterogeneity'], ordered=True)
# lfs['Outcome'] = pd.Categorical(lfs['Outcome'], categories=[COL_PRETTY_NAMES.get(o, o) for o in ['plasticity_tv', 'monotonicity', 'influence_out', 'modal_consensus_change', 'modal_consensus_T']], ordered=True)
# lfs= lfs.sort_values(['Outcome', 'Contrast']).set_index(['Outcome', 'Contrast'])



# print(lfs.to_latex(index=True,
#                 caption="Marginal contrasts on key outcomes.",
#                 multicolumn  = True,
#                 multirow     = True,
#                 escape       = False,
#                 float_format="{:0.2f}".format,
#                 label="tab:marginal-contrasts"))
    


### Conditional

In [ ]:
contrasts_latex

In [ ]:


contrast_labels = {
        "role_eff": "Role Effect",
        "model_eff": "Model Effect",
        "alignment": "Role Alignment",
        "dissociation": "Heterogeneity",
    }

graph_labels = {
    "erdos-renyi": "ER",
    "watts-strogatz": "WS",
}

def fmt_p(p: float) -> str:
    if pd.isna(p):
        return ""
    return "<0.001" if p < 0.001 else f"{p:.3f}"


lfs  = (contrasts_latex.filter(pl.col('contrast_type') == 'within')
 .with_columns(pl.col('outcome').map_elements(lambda x: COL_PRETTY_NAMES.get(x, x)).alias('Outcome'))
 .with_columns(pl.col('contrast').map_elements(lambda x: contrast_labels.get(x, x)).alias('Contrast'))
 .with_columns(pl.col('graph_type').map_elements(lambda x: graph_labels.get(x, x)).alias('Network'))
 .select(
     pl.col('Outcome'),
     pl.col('Contrast'),
     pl.col('Network'),
     pl.col('estimate').round(3),
     pl.col('lower').round(3),
     pl.col('upper').round(3),
     pl.col('SE').round(3),
     pl.col('t').round(3),
     pl.col('p').map_elements(fmt_p).alias('p'),
     pl.col('df').round(3),
 )
 ).sort(['Outcome', 'Contrast', 'Network']).to_pandas()
lfs['Contrast'] = pd.Categorical(lfs['Contrast'], categories=['Role Effect', 'Model Effect', 'Role Alignment', 'Heterogeneity'], ordered=True)
lfs['Outcome'] = pd.Categorical(lfs['Outcome'], categories=[COL_PRETTY_NAMES.get(o, o) for o in ['plasticity_tv', 'monotonicity', 'influence_out', 'modal_consensus_change', 'modal_consensus_T']], ordered=True)
lfs= lfs.sort_values(['Outcome', 'Network', 'Contrast']).set_index(['Outcome', 'Network', 'Contrast'])



print(lfs.to_latex(index=True,
                caption="Marginal contrasts on key outcomes.",
                multicolumn  = True,
                multirow     = True,
                escape       = False,
                float_format="{:0.2f}".format,
                label="tab:marginal-contrasts"))
    


In [ ]:
fig = plot_contrast_forest(contrasts[contrasts['outcome'].isin(COL_PRETTY_NAMES.keys())], 
                     use_standardized=True,
                     figsize=(11, 3.5))
fig.savefig(result_path/ "contrasts" / "contrast_forest.pdf", dpi=300, bbox_inches="tight")
plt.show()